<a href="https://colab.research.google.com/github/Divyansh-Sharma-021/Student-Performance-Analytics-Engine-/blob/main/Student_Performance_Analytics_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

def show_table(title, table, decimals=2):
    print(f"\n{title}")
    print("=" * len(title))
    print(table.to_string(float_format=lambda value: f"{value:.{decimals}f}"))

try:
    from google.colab import files
except ImportError:
    files = None

In [4]:
rng = np.random.default_rng(seed=42)
marks = rng.normal(loc=65, scale=12, size=(500, 6))
marks = np.clip(marks, 0, 100)
marks = np.round(marks).astype(int)

subjects = ["Maths", "Physics", "Chemistry", "English", "Hindi", "CS"]
student_ids = np.arange(1, 501)

print("STUDENT PERFORMANCE ANALYTICS")
print("=" * 80)
print(f"Students: {marks.shape[0]}  |  Subjects: {marks.shape[1]}  |  Matrix: {marks.shape}")

marks_table = pd.DataFrame(marks[:5], index=student_ids[:5], columns=subjects)
marks_table.index.name = "Student"
show_table("FIRST 5 STUDENTS' MARKS", marks_table, decimals=0)

STUDENT PERFORMANCE ANALYTICS
Students: 500  |  Subjects: 6  |  Matrix: (500, 6)

FIRST 5 STUDENTS' MARKS
         Maths  Physics  Chemistry  English  Hindi  CS
Student                                               
1           69       53         74       76     42  49
2           67       61         65       55     76  74
3           66       79         71       55     69  53
4           76       64         63       57     80  63
5           60       61         71       69     70  70


In [5]:
subject_stats = pd.DataFrame(
    {
        "Sum": np.sum(marks, axis=0),
        "Mean": np.mean(marks, axis=0),
        "Median": np.median(marks, axis=0),
        "Std": np.std(marks, axis=0),
        "Variance": np.var(marks, axis=0),
    },
    index=subjects,
)
subject_stats.index.name = "Subject"
show_table("SUBJECT STATISTICS", subject_stats)


SUBJECT STATISTICS
             Sum  Mean  Median   Std  Variance
Subject                                       
Maths      32245 64.49   65.00 12.17    148.04
Physics    32231 64.46   65.00 12.49    155.99
Chemistry  32214 64.43   64.00 12.33    151.91
English    32664 65.33   65.00 12.13    147.25
Hindi      32404 64.81   65.00 11.45    131.06
CS         32341 64.68   66.00 11.93    142.30


In [6]:
student_sums = np.sum(marks, axis=1)
student_means = np.mean(marks, axis=1)
student_medians = np.median(marks, axis=1)
student_stds = np.std(marks, axis=1)
student_variances = np.var(marks, axis=1)

student_stats = pd.DataFrame(
    {
        "Total": student_sums[:5],
        "Mean": student_means[:5],
        "Median": student_medians[:5],
        "Std": student_stds[:5],
        "Variance": student_variances[:5],
    },
    index=student_ids[:5],
)
student_stats.index.name = "Student"
show_table("FIRST 5 STUDENTS' STATISTICS", student_stats)


FIRST 5 STUDENTS' STATISTICS
         Total  Mean  Median   Std  Variance
Student                                     
1          363 60.50   61.00 13.07    170.92
2          398 66.33   66.00  7.20     51.89
3          393 65.50   67.50  9.05     81.92
4          403 67.17   63.50  8.07     65.14
5          401 66.83   69.50  4.52     20.47


In [7]:
subject_means = np.mean(marks, axis=0)
subject_stds = np.std(marks, axis=0)
z_scores = (marks - subject_means) / subject_stds
outlier_mask = np.abs(z_scores) > 2

outlier_counts = pd.DataFrame(
    {"Outlier marks": np.sum(outlier_mask, axis=0)}, index=subjects
)
outlier_counts.index.name = "Subject"
show_table("OUTLIERS BY SUBJECT (|Z| > 2)", outlier_counts, decimals=0)
print("Total outlier marks:", int(np.sum(outlier_mask)))

outlier_rows, outlier_columns = np.where(outlier_mask)
outlier_sample = pd.DataFrame(
    {
        "Student": student_ids[outlier_rows[:10]],
        "Subject": [subjects[column] for column in outlier_columns[:10]],
        "Mark": marks[outlier_rows[:10], outlier_columns[:10]],
        "Z-score": z_scores[outlier_rows[:10], outlier_columns[:10]],
    }
)
show_table("FIRST 10 OUTLIER MARKS", outlier_sample)


OUTLIERS BY SUBJECT (|Z| > 2)
           Outlier marks
Subject                 
Maths                 25
Physics               23
Chemistry             21
English               29
Hindi                 24
CS                    19
Total outlier marks: 141

FIRST 10 OUTLIER MARKS
   Student    Subject  Mark  Z-score
0        6      Maths    91     2.18
1       24      Maths    89     2.01
2       24    Physics   100     2.85
3       24      Hindi    39    -2.25
4       28      Maths    91     2.18
5       35      Maths    40    -2.01
6       38      Hindi    95     2.64
7       40  Chemistry    39    -2.06
8       41      Hindi    91     2.29
9       42  Chemistry   100     2.89


In [8]:
order = np.argsort(student_sums)[::-1]
ranks = np.empty(500, dtype=int)
ranks[order] = np.arange(1, 501)

top_students = pd.DataFrame(
    {
        "Rank": ranks[order[:10]],
        "Student": student_ids[order[:10]],
        "Total": student_sums[order[:10]],
        "Average": student_means[order[:10]],
    }
)
show_table("TOP 10 STUDENTS", top_students)


TOP 10 STUDENTS
   Rank  Student  Total  Average
0     1      499    466    77.67
1     2      166    459    76.50
2     3      418    456    76.00
3     4      129    451    75.17
4     5      184    451    75.17
5     6      390    451    75.17
6     7      369    450    75.00
7     8       52    449    74.83
8     9      472    446    74.33
9    10      383    446    74.33


In [11]:
student_ids_column = student_ids.reshape(-1, 1)
totals_column = student_sums.reshape(-1, 1)
ranks_column = ranks.reshape(-1, 1)
final_table = np.hstack((student_ids_column, marks, totals_column, ranks_column))

report_columns = ["Student_ID"] + subjects + ["Total", "Rank"]
report = pd.DataFrame(final_table, columns=report_columns)
filename = "student_performance_.csv"
report.to_csv(filename, index=False)

hardest_subject = subjects[int(np.argmin(subject_means))]
easiest_subject = subjects[int(np.argmax(subject_means))]
most_inconsistent_subject = subjects[int(np.argmax(subject_stds))]

print("\nSUMMARY")
print("-" * 80)
print(f"Hardest subject (lowest mean):       {hardest_subject}")
print(f"Easiest subject (highest mean):      {easiest_subject}")
print(f"Most inconsistent (highest std):     {most_inconsistent_subject}")
print(f"Top student:                         Student {student_ids[order[0]]}")
print(f"Top total:                           {student_sums[order[0]]}")
print(f"Outlier marks:                       {int(np.sum(outlier_mask))}")
print(f"CSV report shape:                    {report.shape}")
print(f"CSV saved as:                        {filename}")

if files is not None:
    files.download(filename)
else:
    print("Run this file in Google Colab to download the CSV automatically.")


SUMMARY
--------------------------------------------------------------------------------
Hardest subject (lowest mean):       Chemistry
Easiest subject (highest mean):      English
Most inconsistent (highest std):     Physics
Top student:                         Student 499
Top total:                           466
Outlier marks:                       141
CSV report shape:                    (500, 9)
CSV saved as:                        student_performance_.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>